# CICIoT2023 Benchmark (Demo-Safe)\n
\n
This notebook reimplements the paper benchmark workflow on the provided CSV feature data.\n
\n
Run order for this notebook:\n
1. Binary classification (Benign vs Attack)\n
2. Grouped 8-class classification\n
3. Fine-grained multi-class classification\n
\n
Local safety policy in this notebook:\n
- Only a few files per run\n
- Row cap per file\n
- Save artifacts after every run

## Step 1: Imports and Paths\n
Decision: keep all reusable training logic in src/cic_iot_benchmark.py so each run is repeatable.

In [ ]:
from pathlib import Path\n
import sys\n
\n
PROJECT_ROOT = Path('..').resolve()\n
if str(PROJECT_ROOT) not in sys.path:\n
    sys.path.insert(0, str(PROJECT_ROOT))\n
\n
from src.cic_iot_benchmark import (\n
    DemoConfig,\n
    discover_merged_files,\n
    ensure_output_dirs,\n
    infer_label_column,\n
    load_capped_data,\n
    run_benchmark_task,\n
    save_run_artifacts,\n
    set_global_seed,\n
)

## Step 2: Safe Demo Configuration\n
Decision: start small to avoid local crashes. Increase one dimension at a time (rows or file count), never both at once.

In [ ]:
cfg = DemoConfig(\n
    data_dir=PROJECT_ROOT / 'data' / 'CIC_IOT_Dataset_2023',\n
    max_files=2,\n
    rows_per_file=25000,\n
    test_size=0.2,\n
    random_state=42,\n
)\n
\n
set_global_seed(cfg.random_state)\n
output_dirs = ensure_output_dirs(PROJECT_ROOT / 'outputs')\n
\n
print('Config:', cfg)\n
print('Output root:', output_dirs['base'])

## Step 3: Load Capped Data and Infer Label Column\n
Decision: use merged files only and add source file markers for traceability in metrics artifacts.

In [ ]:
merged_files = discover_merged_files(cfg.data_dir)\n
print(f'Merged CSV files discovered: {len(merged_files)}')\n
\n
df, selected_files = load_capped_data(\n
    merged_files,\n
    max_files=cfg.max_files,\n
    rows_per_file=cfg.rows_per_file,\n
)\n
\n
label_col = infer_label_column(df)\n
print('Selected files:', [p.name for p in selected_files])\n
print('Sample shape:', df.shape)\n
print('Inferred label column:', label_col)\n
df.head(3)

## Step 4: Binary Benchmark First\n
Decision: run binary first because it is the lowest-risk sanity check for the full pipeline.

In [ ]:
binary_results, binary_models, binary_reports, y_binary, dropped_binary = run_benchmark_task(\n
    df=df,\n
    label_col=label_col,\n
    task='binary',\n
    test_size=cfg.test_size,\n
    random_state=cfg.random_state,\n
)\n
\n
binary_run_dir = save_run_artifacts(\n
    output_root=output_dirs['base'],\n
    task='binary',\n
    result_df=binary_results,\n
    reports=binary_reports,\n
    models=binary_models,\n
    selected_files=selected_files,\n
    label_distribution=y_binary.value_counts(dropna=False),\n
    dropped_columns=dropped_binary,\n
    feature_columns=list(binary_models[next(iter(binary_models))].named_steps.get('scaler', None).feature_names_in_) if hasattr(binary_models[next(iter(binary_models))].named_steps.get('scaler', None), 'feature_names_in_') else [],\n
)\n
\n
print('Artifacts saved to:', binary_run_dir)\n
binary_results

## Step 5: Grouped 8-Class Benchmark\n
Decision: use the same training/eval pipeline as binary so only the target mapping changes.

In [ ]:
grouped_results, grouped_models, grouped_reports, y_grouped, dropped_grouped = run_benchmark_task(\n
    df=df,\n
    label_col=label_col,\n
    task='grouped_8',\n
    test_size=cfg.test_size,\n
    random_state=cfg.random_state,\n
)\n
\n
grouped_run_dir = save_run_artifacts(\n
    output_root=output_dirs['base'],\n
    task='grouped_8',\n
    result_df=grouped_results,\n
    reports=grouped_reports,\n
    models=grouped_models,\n
    selected_files=selected_files,\n
    label_distribution=y_grouped.value_counts(dropna=False),\n
    dropped_columns=dropped_grouped,\n
    feature_columns=list(grouped_models[next(iter(grouped_models))].named_steps.get('scaler', None).feature_names_in_) if hasattr(grouped_models[next(iter(grouped_models))].named_steps.get('scaler', None), 'feature_names_in_') else [],\n
)\n
\n
print('Artifacts saved to:', grouped_run_dir)\n
grouped_results

## Step 6: Fine-Grained (34-Class Style) Benchmark\n
Decision: this is usually the hardest task, so run it after pipeline confidence is established.

In [ ]:
fine_results, fine_models, fine_reports, y_fine, dropped_fine = run_benchmark_task(\n
    df=df,\n
    label_col=label_col,\n
    task='fine_34',\n
    test_size=cfg.test_size,\n
    random_state=cfg.random_state,\n
)\n
\n
fine_run_dir = save_run_artifacts(\n
    output_root=output_dirs['base'],\n
    task='fine_34',\n
    result_df=fine_results,\n
    reports=fine_reports,\n
    models=fine_models,\n
    selected_files=selected_files,\n
    label_distribution=y_fine.value_counts(dropna=False),\n
    dropped_columns=dropped_fine,\n
    feature_columns=list(fine_models[next(iter(fine_models))].named_steps.get('scaler', None).feature_names_in_) if hasattr(fine_models[next(iter(fine_models))].named_steps.get('scaler', None), 'feature_names_in_') else [],\n
)\n
\n
print('Artifacts saved to:', fine_run_dir)\n
fine_results

## Step 7: What to Scale Next\n
Use this controlled progression to avoid crashes:\n
1. Increase rows_per_file (keep max_files fixed)\n
2. Increase max_files (keep rows_per_file fixed)\n
3. Never increase both in one jump